In [ ]:
import numpy as np
from IPython.display import clear_output
import time
import random
# https://www.youtube.com/watch?v=UXW2yZndl7U

In [ ]:
def update_board(board_temp,color,column):
    # this is a function that takes the current board status, a color, and a column and outputs the new board status
    # columns 0 - 6 are for putting a checker on the board: if column is full just return the current board...this should be forbidden by the player

    # the color input should be either 'plus' or 'minus'

    board = board_temp.copy()
    ncol = board.shape[1]
    nrow = board.shape[0]

    # this seems silly, but actually faster to run than using sum because of overhead!
    colsum = abs(board[0,column])+abs(board[1,column])+abs(board[2,column])+abs(board[3,column])+abs(board[4,column])+abs(board[5,column])
    row = int(5-colsum)
    if row > -0.5:
        if color == 'plus':
            board[row,column] = 1
        else:
            board[row,column] = -1
    return board

# in this code the board is a 6x7 numpy array.  Each entry is +1, -1 or 0.  You WILL be able to do a better
# job training your neural network if you rearrange this to be a 6x7x2 numpy array.  If the i'th row and j'th
# column is +1, this can be represented by board[i,j,0]=1.  If it is -1, this can be represented by
# board[i,j,1]=1. It's up to you how you represent your board.


In [ ]:
board = np.zeros((6,7))
board = update_board(board,'plus',3)
board = update_board(board,'minus',3)
board = update_board(board,'plus',3)
board = update_board(board,'minus',3)
board = update_board(board,'plus',3)
board = update_board(board,'minus',3)
print(board)
board = update_board(board,'plus',3)
print(board)

[[ 0.  0.  0. -1.  0.  0.  0.]
 [ 0.  0.  0.  1.  0.  0.  0.]
 [ 0.  0.  0. -1.  0.  0.  0.]
 [ 0.  0.  0.  1.  0.  0.  0.]
 [ 0.  0.  0. -1.  0.  0.  0.]
 [ 0.  0.  0.  1.  0.  0.  0.]]
[[ 0.  0.  0. -1.  0.  0.  0.]
 [ 0.  0.  0.  1.  0.  0.  0.]
 [ 0.  0.  0. -1.  0.  0.  0.]
 [ 0.  0.  0.  1.  0.  0.  0.]
 [ 0.  0.  0. -1.  0.  0.  0.]
 [ 0.  0.  0.  1.  0.  0.  0.]]


In [ ]:
def check_for_win_slow(board):
    # this function checks to see if anyone has won on the given board
    nrow = board.shape[0]
    ncol = board.shape[1]
    winner = 'nobody'
    for col in range(ncol):
        for row in reversed(range(nrow)):
            if abs(board[row,col]) < 0.1: # if this cell is empty, all the cells above it are too!
                break
            # check for vertical winners
            if row <= (nrow-4): # can't have a column go from rows 4-7...
                tempsum = board[row,col]+board[row+1,col]+board[row+2,col]+board[row+3,col] # this is WAY faster than np.sum!!!
                if tempsum==4:
                    winner = 'v-plus'
                    return winner
                elif tempsum==-4:
                    winner = 'v-minus'
                    return winner
            # check for horizontal winners
            if col <= (ncol-4):
                tempsum = board[row,col]+board[row,col+1]+board[row,col+2]+board[row,col+3]
                if tempsum==4:
                    winner = 'h-plus'
                    return winner
                elif tempsum==-4:
                    winner = 'h-minus'
                    return winner
            # check for top left to bottom right diagonal winners
            if (row <= (nrow-4)) and (col <= (ncol-4)):
                tempsum = board[row,col]+board[row+1,col+1]+board[row+2,col+2]+board[row+3,col+3]
                if tempsum==4:
                    winner = 'd-plus'
                    return winner
                elif tempsum==-4:
                    winner = 'd-minus'
                    return winner
            # check for top right to bottom left diagonal winners
            if (row <= (nrow-4)) and (col >= 3):
                tempsum = board[row,col]+board[row+1,col-1]+board[row+2,col-2]+board[row+3,col-3]
                if tempsum==4:
                    winner = 'd-plus'
                    return winner
                elif tempsum==-4:
                    winner = 'd-minus'
                    return winner
    return winner

In [ ]:
def check_for_win(board,col):
    # this code is faster than the above code, but it requires knowing where the last checker was dropped
    # it may seem extreme, but in MCTS this function is called more than anything and actually makes up
    # a large portion of total time spent finding a good move.  So every microsecond is worth saving!
    nrow = 6
    ncol = 7
    # take advantage of knowing what column was last played in...need to check way fewer possibilities
    colsum = abs(board[0,col])+abs(board[1,col])+abs(board[2,col])+abs(board[3,col])+abs(board[4,col])+abs(board[5,col])
    row = int(6-colsum)
    if row+3<6:
        vert = board[row,col] + board[row+1,col] + board[row+2,col] + board[row+3,col]
        if vert == 4:
            return 'v-plus'
        elif vert == -4:
            return 'v-minus'
    if col+3<7:
        hor = board[row,col] + board[row,col+1] + board[row,col+2] + board[row,col+3]
        if hor == 4:
            return 'h-plus'
        elif hor == -4:
            return 'h-minus'
    if col-1>=0 and col+2<7:
        hor = board[row,col-1] + board[row,col] + board[row,col+1] + board[row,col+2]
        if hor == 4:
            return 'h-plus'
        elif hor == -4:
            return 'h-minus'
    if col-2>=0 and col+1<7:
        hor = board[row,col-2] + board[row,col-1] + board[row,col] + board[row,col+1]
        if hor == 4:
            return 'h-plus'
        elif hor == -4:
            return 'h-minus'
    if col-3>=0:
        hor = board[row,col-3] + board[row,col-2] + board[row,col-1] + board[row,col]
        if hor == 4:
            return 'h-plus'
        elif hor == -4:
            return 'h-minus'
    if row < 3 and col < 4:
        DR = board[row,col] + board[row+1,col+1] + board[row+2,col+2] + board[row+3,col+3]
        if DR == 4:
            return 'd-plus'
        elif DR == -4:
            return 'd-minus'
    if row-1>=0 and col-1>=0 and row+2<6 and col+2<7:
        DR = board[row-1,col-1] + board[row,col] + board[row+1,col+1] + board[row+2,col+2]
        if DR == 4:
            return 'd-plus'
        elif DR == -4:
            return 'd-minus'
    if row-2>=0 and col-2>=0 and row+1<6 and col+1<7:
        DR = board[row-2,col-2] + board[row-1,col-1] + board[row,col] + board[row+1,col+1]
        if DR == 4:
            return 'd-plus'
        elif DR == -4:
            return 'd-minus'
    if row-3>=0 and col-3>=0:
        DR = board[row-3,col-3] + board[row-2,col-2] + board[row-1,col-1] + board[row,col]
        if DR == 4:
            return 'd-plus'
        elif DR == -4:
            return 'd-minus'
    if row+3<6 and col-3>=0:
        DL = board[row,col] + board[row+1,col-1] + board[row+2,col-2] + board[row+3,col-3]
        if DL == 4:
            return 'd-plus'
        elif DL == -4:
            return 'd-minus'
    if row-1 >= 0 and col+1 < 7 and row+2<6 and col-2>=0:
        DL = board[row-1,col+1] + board[row,col] + board[row+1,col-1] + board[row+2,col-2]
        if DL == 4:
            return 'd-plus'
        elif DL == -4:
            return 'd-minus'
    if row-2 >=0 and col+2<7 and row+1<6 and col-1>=0:
        DL = board[row-2,col+2] + board[row-1,col+1] + board[row,col] + board[row+1,col-1]
        if DL == 4:
            return 'd-plus'
        elif DL == -4:
            return 'd-minus'
    if row-3>=0 and col+3<7:
        DL = board[row-3,col+3] + board[row-2,col+2] + board[row-1,col+1] + board[row,col]
        if DL == 4:
            return 'd-plus'
        elif DL == -4:
            return 'd-minus'
    return 'nobody'

In [ ]:
def find_legal(board):
    legal = [i for i in range(7) if abs(board[0,i]) < 0.1]
    return legal

In [ ]:
def look_for_win(board_,color):
    board_ = board_.copy()
    legal = find_legal(board_)
    winner = -1
    for m in legal:
        bt = update_board(board_.copy(),color,m)
        wi = check_for_win(bt,m)
        if wi[2:] == color:
            winner = m
            break
    return winner

In [ ]:
def find_all_nonlosers(board,color):
    if color == 'plus':
        opp = 'minus'
    else:
        opp = 'plus'
    legal = find_legal(board)
    poss_boards = [update_board(board,color,l) for l in legal]
    poss_legal = [find_legal(b) for b in poss_boards]
    allowed = []
    for i in range(len(legal)):
        wins = [j for j in poss_legal[i] if check_for_win(update_board(poss_boards[i],opp,j),j) != 'nobody']
        if len(wins) == 0:
            allowed.append(legal[i])
    return allowed

In [ ]:
def back_prop(winner,path,color0,md):
    for i in range(len(path)):
        board_temp = path[i]

        md[board_temp][0]+=1
        if winner[2]==color0[0]:
            if i % 2 == 1:
                md[board_temp][1] += 1
            else:
                md[board_temp][1] -= 1
        elif winner[2]=='e': # tie
            # md[board_temp][1] += 0
            pass
        else:
            if i % 2 == 1:
                md[board_temp][1] -= 1
            else:
                md[board_temp][1] += 1

In [ ]:
def rollout(board,next_player):
    winner = 'nobody'
    player = next_player
    while winner == 'nobody':
        legal = find_legal(board)
        if len(legal) == 0:
            winner = 'tie'
            return winner
        move = random.choice(legal)
        board = update_board(board,player,move)
        winner = check_for_win(board,move)

        if player == 'plus':
            player = 'minus'
        else:
            player = 'plus'
    return winner


In [ ]:
def mcts(board_temp,color0,nsteps):
    # nsteps is a parameter that determines the skill (and slowness) of the player
    # bigger values of nsteps means the player is better, but also slower to figure out a move.
    board = board_temp.copy()
    ##############################################
    winColumn = look_for_win(board,color0) # check to find a winning column
    if winColumn > -0.5:
        return winColumn # if there is one - play that!
    legal0 = find_all_nonlosers(board,color0) # find all moves that won't immediately lead to your opponent winning
    if len(legal0) == 0: # if you can't block your opponent - just find the 'best' losing move
        legal0 = find_legal(board)
    ##############################################
    # the code above, in between the hash rows, is not part of traditional MCTS
    # but it makes it better and faster - so I included it!
    # MCTS occasionally makes stupid mistakes
    # like not dropping the checker on a winning column, or not blocking an obvious opponent win
    # this avoids a little bit of that stupidity!
    # we could also add this logic to the rest of the MCTS and rollout functions - I just haven't done that yet...
    # feel free to experiment!
    mcts_dict = {tuple(board.ravel()):[0,0]}
    for ijk in range(nsteps):
        color = color0
        winner = 'nobody'
        board_mcts = board.copy()
        path = [tuple(board_mcts.ravel())]
        while winner == 'nobody':
            legal = find_legal(board_mcts)
            if len(legal) == 0:
                winner = 'tie'
                back_prop(winner,path,color0,mcts_dict)
                break
            board_list = []
            for col in legal:
                board_list.append(tuple(update_board(board_mcts,color,col).ravel()))
            for bl in board_list:
                if bl not in mcts_dict.keys():
                    mcts_dict[bl] = [0,0]
            ucb1 = np.zeros(len(legal))
            for i in range(len(legal)):
                num_denom = mcts_dict[board_list[i]]
                if num_denom[0] == 0:
                    ucb1[i] = 10*nsteps
                else:
                    ucb1[i] = num_denom[1]/num_denom[0] + 2*np.sqrt(np.log(mcts_dict[path[-1]][0])/mcts_dict[board_list[i]][0])
            chosen = np.argmax(ucb1)

            board_mcts = update_board(board_mcts,color,legal[chosen])
            path.append(tuple(board_mcts.ravel()))
            winner = check_for_win(board_mcts,legal[chosen])
            if winner[2]==color[0]:
                back_prop(winner,path,color0,mcts_dict)
                break
            if color == 'plus':
                color = 'minus'
            else:
                color = 'plus'
            if mcts_dict[tuple(board_mcts.ravel())][0] == 0:
                winner = rollout(board_mcts,color)
                back_prop(winner,path,color0,mcts_dict)
                break

    maxval = -np.inf
    best_col = -1
    for col in legal0:
        board_temp = tuple(update_board(board,color0,col).ravel())
        num_denom = mcts_dict[board_temp]
        if num_denom[0] == 0:
            compare = -np.inf
        else:
            compare = num_denom[1] / num_denom[0]
        if compare > maxval:
            maxval = compare
            best_col = col
    return (best_col)


In [ ]:
mcts(np.zeros((6,7)),'plus',5000)

3

In [ ]:
import pandas as pd
import random
import time
import numpy as np
import os
from google.colab import drive

# --- 1. MOUNT GOOGLE DRIVE ---
# This ensures files are saved to Drive, not just the temporary Colab storage.
drive.mount('/content/drive')

# Set the path to save in your Drive (Modify 'My Drive' subfolder if needed)
file_path = '/content/drive/My Drive/connect4_data.npz'

# --- CONFIGURATION ---
games_to_play = 1000  # Total target games
min_steps = 500
max_steps = 5000
checkpoint_interval = 50

# --- 2. CRASH RECOVERY (LOAD PREVIOUS DATA) ---
data_boards = []
data_moves = []
games_already_played = 0

if os.path.exists(file_path):
    print(f"Found existing data at {file_path}. Loading to resume...")
    loaded_data = np.load(file_path)
    # Load 'X' and 'y' back into lists
    data_boards = list(loaded_data['X'])
    data_moves = list(loaded_data['y'])

    # Calculate how many unique games we likely have (approx /2 because of mirroring)
    games_already_played = len(data_moves) // 2
    print(f"Resuming from approximately Game {games_already_played}. Total samples so far: {len(data_moves)}")
else:
    print("No existing data found. Starting fresh.")

print(f"Targeting {games_to_play} total games...")

start_time = time.time()

# Loop from current progress up to target
for i in range(games_already_played, games_to_play):
    board = np.zeros((6,7))
    winner = 'nobody'
    player = 'plus'
    move_count = 0
    random_moves_count = random.randint(1, 6)

    while winner == 'nobody':
        # DECIDE MOVE
        if move_count < random_moves_count:
            # Play random move (Don't save)
            legal_moves = [c for c in range(7) if board[0,c] == 0]
            if not legal_moves: break
            col = random.choice(legal_moves)
            is_training_data = False
        else:
            # Play MCTS move (Save)
            current_steps = random.randint(min_steps, max_steps)
            col = mcts(board, player, current_steps)
            is_training_data = True

        # SAVE DATA (+ REFLECTION)
        if is_training_data:
            # 1. PREPARE THE RAW BOARD (Player Perspective)
            if player == 'plus':
                current_board_state = board.copy()
            else:
                current_board_state = board.copy() * -1

            # 2. SAVE ORIGINAL (No .ravel() here, keeping it 6x7)
            data_boards.append(current_board_state)
            data_moves.append(col)

            # 3. SAVE MIRROR IMAGE
            # Flip board horizontally (keeps shape 6x7)
            mirrored_board = np.fliplr(current_board_state)
            data_boards.append(mirrored_board)

            # Flip the move
            mirrored_move = 6 - col
            data_moves.append(mirrored_move)

        # UPDATE BOARD
        board = update_board(board, player, col)
        winner = check_for_win(board, col)

        # SWITCH TURNS
        if player == 'plus':
            player = 'minus'
        else:
            player = 'plus'
        move_count += 1

        if move_count == 42 and winner == 'nobody':
            winner = 'tie'

    # --- TIME ESTIMATION ---
    games_finished = i + 1
    # Adjust time calc to only count this session
    elapsed_total = time.time() - start_time
    games_this_session = games_finished - games_already_played

    if games_this_session > 0:
        avg_time_per_game = elapsed_total / games_this_session
        remaining_games = games_to_play - games_finished
        est_remaining_seconds = avg_time_per_game * remaining_games

        if est_remaining_seconds > 3600:
            est_str = f"{est_remaining_seconds/3600:.2f} hours"
        else:
            est_str = f"{est_remaining_seconds/60:.1f} minutes"
    else:
        est_str = "Calculating..."

    print(f"Game {games_finished}/{games_to_play} finished. MCTS Steps {current_steps}. Winner: {winner}. Total samples: {len(data_moves)}. Est Wait: {est_str}")

    # --- CHECKPOINT SAVE (.npz) ---
    if games_finished % checkpoint_interval == 0:
        print(f"--- CHECKPOINT: Saving {len(data_moves)} samples to Drive... ---")

        # Convert lists to numpy arrays matching the requested shape
        X_array = np.array(data_boards, dtype=np.float64) # Shape (N, 6, 7)
        y_array = np.array(data_moves, dtype=np.int64)    # Shape (N,)

        # Save as compressed npz to save space
        np.savez_compressed(file_path, X=X_array, y=y_array)
        print("--- Save Complete. Continuing... ---")

# FINAL SAVE
print("Final Save...")
X_array = np.array(data_boards, dtype=np.float64)
y_array = np.array(data_moves, dtype=np.int64)
np.savez_compressed(file_path, X=X_array, y=y_array)
print(f"Done! Saved {len(X_array)} samples to '{file_path}'.")

MessageError: Error: credential propagation was unsuccessful

## **CNN**

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, optimizers, callbacks
import os

# ==========================================
# DATA LOADING & PREPROCESSING
# ==========================================
file_path = 'dean800k.npz'

if os.path.exists(file_path):
    print(f"Loading {file_path}...")
    with np.load(file_path) as data:
        X_raw = data['X']  # Shape: (N, 6, 7)
        y = data['pi']     # Shape: (N, 7) - Probability distribution

    # Convert to (N, 6, 7, 2) for dual channels (Player/Opponent)
    X_processed = np.zeros((X_raw.shape[0], 6, 7, 2), dtype=np.float32)
    X_processed[..., 0] = (X_raw == 1).astype(np.float32) #bitboard player 1
    X_processed[..., 1] = (X_raw == -1).astype(np.float32) #bitboard player 1

    """
    Neural networks struggle with single inputs that represent opposing concepts (like 1 vs -1 vs 0).
    By splitting them into two separate "maps," the network can independently learn patterns like
    "My pieces are forming a line" vs "Opponent pieces are blocking me."

    """


    print(f"Input Shape: {X_processed.shape}")
    print(f"Target Shape: {y.shape}")

else:
    print("Error: 'dean800k.npz' not found.")
    X_processed, y = None, None

# ==========================================
# RESNET ARCHITECTURE
# ==========================================
def res_layer(tensor, filters, kernel_size=(3,3), reg_strength=1e-5):
    """
    Creates a residual block with ELU activations for better gradient flow.
    Structure: Conv -> BN -> ELU -> Conv -> BN -> ADD -> ELU
    """
    shortcut = tensor

    # First Convolution
    x = layers.Conv2D(filters, kernel_size, padding='same',
                      kernel_regularizer=regularizers.l2(reg_strength))(tensor)
    x = layers.BatchNormalization()(x)
    x = layers.ELU()(x)  # ELU handles vanishing gradients better than ReLU

    # Second Convolution
    x = layers.Conv2D(filters, kernel_size, padding='same',
                      kernel_regularizer=regularizers.l2(reg_strength))(x)
    x = layers.BatchNormalization()(x)

    # Residual Connection (Skip)
    x = layers.Add()([shortcut, x])
    x = layers.ELU()(x)

    return x

def create_advanced_cnn(input_shape, num_outputs, l2_reg=1e-5, dropout_rate=0.1):
    """
    Builds a ResNet-style model adapted for the small 6x7 board.
    Improvement: Removed MaxPooling layers (found in standard ResNets) because
    the board is too small (6x7) to afford downsampling spatial dimensions.
    """
    inputs = layers.Input(shape=input_shape)

    # Initial Feature Extraction
    x = layers.Conv2D(128, (3,3), padding='same',
                      kernel_regularizer=regularizers.l2(l2_reg))(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ELU()(x)

    # Deep Residual Tower (5 Blocks)
    # Keeping filters constant at 128 preserves board information
    for _ in range(5):
        x = res_layer(x, filters=128, reg_strength=l2_reg)

    # Policy Head
    x = layers.Flatten()(x)

    x = layers.Dense(1024, kernel_regularizer=regularizers.l2(l2_reg))(x)
    x = layers.BatchNormalization()(x)
    x = layers.ELU()(x)
    x = layers.Dropout(dropout_rate)(x)

    x = layers.Dense(512, kernel_regularizer=regularizers.l2(l2_reg))(x)
    x = layers.BatchNormalization()(x)
    x = layers.ELU()(x)
    x = layers.Dropout(dropout_rate)(x)

    outputs = layers.Dense(num_outputs, activation='softmax')(x)

    return models.Model(inputs, outputs, name="DeepResNet_Connect4")

# ==========================================
# TRAINING
# ==========================================
if X_processed is not None:
    # Build Model
    model = create_advanced_cnn(
        input_shape=(6, 7, 2),
        num_outputs=7,
        l2_reg=1e-5,
        dropout_rate=0.15
    )

    # Compile
    # Note: Using categorical_crossentropy because 'pi' is a probability distribution
    model.compile(
        optimizer=optimizers.Adam(learning_rate=0.0001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    model.summary()

    # Callbacks
    early_stopping = callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    )

    # Fit
    history = model.fit(
        X_processed, y,
        epochs=50,
        batch_size=256,
        validation_split=0.1,
        callbacks=[early_stopping],
        verbose=1
    )

    # Save
    model.save('my_connect4_cnn.keras')
    print("Model saved to my_connect4_cnn.keras")

Loading dean800k.npz...
Input Shape: (840586, 6, 7, 2)
Target Shape: (840586, 7)


Model: "DeepResNet_Connect4"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 6, 7, 2)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 6, 7, 128) │      2,432 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 6, 7, 128) │        512 │ conv2d[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ elu (ELU)           │ (None, 6, 7, 128) │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 6, 7, 128) │    147,584 │ elu[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 6, 7, 128) │        512 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ elu_1 (ELU)         │ (None, 6, 7, 128) │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 6, 7, 128) │    147,584 │ elu_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 6, 7, 128) │        512 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 6, 7, 128) │          0 │ elu[0][0],        │
│                     │                   │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ elu_2 (ELU)         │ (None, 6, 7, 128) │          0 │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 6, 7, 128) │    147,584 │ elu_2[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 6, 7, 128) │        512 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ elu_3 (ELU)         │ (None, 6, 7, 128) │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 6, 7, 128) │    147,584 │ elu_3[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 6, 7, 128) │        512 │ conv2d_4[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 6, 7, 128) │          0 │ elu_2[0][0],      │
│                     │                   │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ elu_4 (ELU)         │ (None, 6, 7, 128) │          0 │ add_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 6, 7, 128) │    147,584 │ elu_4[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 6, 7, 128) │        512 │ conv2d_5[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 7,524,487 (28.70 MB)

 Trainable params: 7,518,599 (28.68 MB)

 Non-trainable params: 5,888 (23.00 KB)

Epoch 1/50
2956/2956 ━━━━━━━━━━━━━━━━━━━━ 122s 36ms/step - accuracy: 0.4040 - loss: 1.6773 - val_accuracy: 0.5690 - val_loss: 1.3611
Epoch 2/50
2956/2956 ━━━━━━━━━━━━━━━━━━━━ 108s 37ms/step - accuracy: 0.5687 - loss: 1.4584 - val_accuracy: 0.5936 - val_loss: 1.3166
Epoch 3/50
2956/2956 ━━━━━━━━━━━━━━━━━━━━ 108s 37ms/step - accuracy: 0.6081 - loss: 1.4130 - val_accuracy: 0.6155 - val_loss: 1.2931
Epoch 4/50
2956/2956 ━━━━━━━━━━━━━━━━━━━━ 108s 37ms/step - accuracy: 0.6301 - loss: 1.3887 - val_accuracy: 0.6327 - val_loss: 1.2831
Epoch 5/50
2956/2956 ━━━━━━━━━━━━━━━━━━━━ 108s 37ms/step - accuracy: 0.6426 - loss: 1.3710 - val_accuracy: 0.6400 - val_loss: 1.2682
Epoch 6/50
2956/2956 ━━━━━━━━━━━━━━━━━━━━ 108s 36ms/step - accuracy: 0.6533 - loss: 1.3597 - val_accuracy: 0.6429 - val_loss: 1.2566
Epoch 7/50
2956/2956 ━━━━━━━━━━━━━━━━━━━━ 108s 37ms/step - accuracy: 0.6609 - loss: 1.3493 - val_accuracy: 0.6544 - val_loss: 1.2492
Epoch 8/50
2956/2956 ━━━━━━━━━━━━━━━━━━━━ 108s 37ms/step - accuracy: 

In [6]:
import numpy as np
import tensorflow as tf
from IPython.display import clear_output
import time

# --- 1. Load the Model ---
try:
    model = tf.keras.models.load_model('my_connect4_cnn.keras')
    print("✅ Model loaded successfully!")
except:
    print("❌ Model file not found. Make sure 'my_connect4_cnn.keras' is in your files.")
    model = None

# --- 2. Game Logic Helper Functions ---
def update_board(board, col, player):
    """Drops a piece into the given column."""
    if board[0, col] != 0:
        return board, False  # Column is full

    new_board = board.copy()
    for row in range(5, -1, -1):
        if new_board[row, col] == 0:
            new_board[row, col] = player
            break
    return new_board, True

def check_win(board, player):
    """Checks if the given player has won."""
    rows, cols = board.shape
    for r in range(rows):
        for c in range(cols):
            if board[r, c] == player:
                # Horizontal
                if c + 3 < cols and all(board[r, c+i] == player for i in range(4)): return True
                # Vertical
                if r + 3 < rows and all(board[r+i, c] == player for i in range(4)): return True
                # Diag Positive
                if r + 3 < rows and c + 3 < cols and all(board[r+i, c+i] == player for i in range(4)): return True
                # Diag Negative
                if r - 3 >= 0 and c + 3 < cols and all(board[r-i, c+i] == player for i in range(4)): return True
    return False

def print_board(board, last_ai_move=None):
    """Visualizes the board with Red/Yellow circles."""
    clear_output()

    # Header
    print("   0    1    2    3    4    5    6 ")
    print(" " + "_____" * 7)

    # Board Rows
    for row in board:
        line = "|"
        for cell in row:
            if cell == 1:    symbol = " 🔴 " # Human
            elif cell == -1: symbol = " 🟡 " # AI
            else:            symbol = " ⚪ " # Empty
            line += symbol + "|"
        print(line)
        print("|" + "_____" * 7 + "|")

    # Footer info
    print("\n🔴 You (Player 1)  vs  🟡 AI (Player 2)")
    if last_ai_move is not None:
        print(f"🤖 AI played in column: {last_ai_move}")

# --- 3. AI Prediction Logic ---
def get_ai_move(model, board, ai_player=-1):
    # Flip board so AI looks like Player 1 for the model
    input_board = board.copy()
    if ai_player == -1:
        input_board = input_board * -1

    X = np.zeros((1, 6, 7, 2))
    X[0, :, :, 0] = (input_board == 1).astype(float)
    X[0, :, :, 1] = (input_board == -1).astype(float)

    probs = model.predict(X, verbose=0)[0]

    # Mask invalid moves
    for col in range(7):
        if board[0, col] != 0:
            probs[col] = -1.0

    return np.argmax(probs)

# --- 4. Main Game Loop ---
if model:
    board = np.zeros((6, 7))
    game_over = False
    last_ai_col = None

    # --- SETUP: WHO GOES FIRST? ---
    print("Welcome!")
    choice = input("Do you want to go first? (y/n): ").lower()
    if choice == 'y':
        turn = 1   # Human starts
        print_board(board)
    else:
        turn = -1  # AI starts
        print("AI is starting...")
        time.sleep(1) # Small pause for effect

    # --- GAME LOOP ---
    while not game_over:
        if turn == 1:
            # === Human Turn ===
            try:
                col = int(input(f"\n👇 Your Move (0-6): "))
                if col < 0 or col > 6: raise ValueError
            except ValueError:
                print("⚠️ Invalid input. Please enter 0-6.")
                continue

            board, success = update_board(board, col, 1)
            if not success:
                print("⚠️ Column full! Try another.")
                continue

            # Update visuals immediately after human moves
            print_board(board, last_ai_move=last_ai_col)

        else:
            # === AI Turn ===
            print("\n🤖 AI is thinking...")
            # Slight delay so it doesn't feel instant/jarring
            time.sleep(0.5)

            col = get_ai_move(model, board, ai_player=-1)
            board, _ = update_board(board, col, -1)
            last_ai_col = col # Save for display

            # Update visuals after AI moves
            print_board(board, last_ai_move=last_ai_col)

        # === Check Status ===
        if check_win(board, turn):
            if turn == 1:
                print("\n🎉🎉 CONGRATULATIONS! YOU WON! 🎉🎉")
            else:
                print("\n🤖 THE AI WON! Better luck next time.")
            game_over = True
        elif np.all(board[0] != 0):
            print("\n⚖️ IT'S A DRAW!")
            game_over = True

        turn *= -1 # Switch turns

   0    1    2    3    4    5    6 
 ___________________________________
| ⚪ | ⚪ | ⚪ | 🟡 | 🟡 | ⚪ | ⚪ |
|___________________________________|
| 🔴 | ⚪ | ⚪ | 🟡 | 🔴 | ⚪ | ⚪ |
|___________________________________|
| 🔴 | ⚪ | 🟡 | 🔴 | 🔴 | ⚪ | 🔴 |
|___________________________________|
| 🟡 | 🟡 | 🔴 | 🟡 | 🟡 | ⚪ | 🟡 |
|___________________________________|
| 🟡 | 🟡 | 🔴 | 🔴 | 🟡 | ⚪ | 🔴 |
|___________________________________|
| 🔴 | 🔴 | 🔴 | 🟡 | 🟡 | 🔴 | 🟡 |
|___________________________________|

🔴 You (Player 1)  vs  🟡 AI (Player 2)
🤖 AI played in column: 1

🤖 THE AI WON! Better luck next time.


In [ ]:
board = np.zeros((6,7))
winner = 'nobody'
color = 'plus'
while winner == 'nobody':
    if color == 'minus':
        col = mcts(board,color,300)
    else:
        col = mcts(board,color,1500)
    board = update_board(board,color,col)
    winner = check_for_win(board,col)
    if color == 'plus':
        color = 'minus'
    else:
        color = 'plus'
    print(board)
    print('=========================')
print(winner)

In [ ]:
import pandas as pd
import numpy as np
import os

# Check if the CSV file exists
if os.path.exists('connect4_data.csv'):
    print("Loading existing data from 'connect4_data.csv'...")
    df_existing = pd.read_csv('connect4_data.csv')

    # Reconstruct data_boards and data_moves from the loaded DataFrame
    # Assuming the last column is 'target_move' and others are board data
    data_moves = df_existing['target_move'].tolist()
    data_boards = df_existing.drop(columns=['target_move']).values.tolist()

    print(f"Loaded {len(data_moves)} existing training examples.")
else:
    print("No existing 'connect4_data.csv' found. Starting fresh.")
    data_boards = []
    data_moves = []

# Now, data_boards and data_moves are populated with any existing data

Loading existing data from 'connect4_data.csv'...
Loaded 29022 existing training examples.


In [ ]:
def display_board(board):
    # this function displays the board as ascii using X for +1 and O for -1
    # For the project, this should be a better picture of the board...
    clear_output()
    horizontal_line = '-'*(7*5+8)
    blank_line = '|'+' '*5
    blank_line *= 7
    blank_line += '|'
    print('   0     1     2     3     4     5     6')
    print(horizontal_line)
    for row in range(6):
        print(blank_line)
        this_line = '|'
        for col in range(7):
            if board[row,col] == 0:
                this_line += ' '*5 + '|'
            elif board[row,col] == 1:
                this_line += '  X  |'
            else:
                this_line += '  O  |'
        print(this_line)
        print(blank_line)
        print(horizontal_line)
    print('   0     1     2     3     4     5     6')



In [ ]:
# this is how you can play a game
winner = 'nobody'
board = np.zeros((6,7))

display_board(board)

player = 'plus'

while winner == 'nobody':
    move = input('Pick a move (0-6) for player '+player+': ')
    move = int(move)
    board = update_board(board,player,move)
    display_board(board)
    winner = check_for_win(board,move)
    if player == 'plus':
        player = 'minus'
    else:
        player = 'plus'
print('The winner is '+winner)
